# Hybrid Quantum Self-Attention Layer Demonstration

This notebook demonstrates our simulator-first implementation of a hybrid quantum self-attention layer for language modeling (1 classical head, 3 quantum heads). We will explore:
1. Loading and tokenizing your actual `input.txt` dataset.
2. Instantiating our Hybrid Quantum Transformer Decoder model.
3. Performing a forward pass and inspecting the output tensor shapes.
4. Training the model with train/validation loss tracking.
5. Visualizing the attention maps to compare classical vs. quantum attention heads.
6. Discussions and Final Conclusions.

---

In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import os
import random

# Import local modules
from config import Hyperparameters as hp
from utils import set_seed, count_parameters
from model import HybridQuantumTransformerDecoder

# Set seeds for reproducibility
set_seed(hp.seed)
print("PyTorch, NumPy, and Qiskit environment loaded successfully!")

## 1. Load and Tokenize `input.txt` Dataset

We read your actual `input.txt` file, build a vocabulary, and prepare training/validation splits!

In [2]:
# 1. Load input.txt
if not os.path.exists('input.txt'):
    import urllib.request
    names_url = 'https://raw.githubusercontent.com/karpathy/makemore/988aa59/names.txt'
    urllib.request.urlretrieve(names_url, 'input.txt')
    
docs = [line.strip() for line in open('input.txt', 'r', encoding='utf-8') if line.strip()]
random.shuffle(docs)
print(f"Number of docs in input.txt: {len(docs)}")

# 2. Build vocabulary from input.txt
uchars = sorted(set(''.join(docs)))
BOS = len(uchars)  # Beginning of sequence token
vocab_size = len(uchars) + 1
print(f"Vocab size: {vocab_size}")
print(f"Unique characters in vocab: {uchars}")

# Character ↔ index mappings
char_to_int = {ch: idx for idx, ch in enumerate(uchars)}
int_to_char = {idx: ch for idx, ch in enumerate(uchars)}
int_to_char[BOS] = '<BOS>'  # For BOS token

# Update hp.vocab_size to match our actual vocab size
hp.vocab_size = vocab_size

# 3. Prepare sliding window data (limit to 100 samples for speed)
def prepare_sliding_window_data(docs, max_samples=100):
    data = []
    for doc in docs:
        # Tokenize the doc with BOS at start
        tokens = [BOS] + [char_to_int[ch] for ch in doc]
        n = min(hp.seq_len, len(tokens))
        # Create sliding window of seq_len
        for start in range(len(tokens) - n + 1):
            if len(data) >= max_samples:
                break
            seq = tokens[start:start + n]
            data.append(seq)
        if len(data) >= max_samples:
            break
    return data

# Prepare data
all_data = prepare_sliding_window_data(docs, max_samples=100)
print(f"Number of sliding window samples: {len(all_data)}")

# Split into train/validation (80/20)
split_idx = int(0.8 * len(all_data))
train_data = all_data[:split_idx]
val_data = all_data[split_idx:]
print(f"Train samples: {len(train_data)}, Val samples: {len(val_data)}")

# Custom Dataset class
class CustomTextDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = torch.tensor(data, dtype=torch.long)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        seq = self.data[idx]
        # Input: tokens[0 : seq_len-1], Target: tokens[1 : seq_len]
        x = seq[:-1]
        y = seq[1:]
        return x, y

# Create DataLoaders
batch_size = 8
train_dataset = CustomTextDataset(train_data)
val_dataset = CustomTextDataset(val_data)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Grab a single batch and decode it
def decode_tokens(tokens):
    """Decodes token IDs back to characters using our input.txt vocabulary."""
    return " ".join([int_to_char.get(t, "?") for t in tokens])

x, y = next(iter(train_loader))
print(f"\nBatch Input Shape: {x.shape} (batch_size, seq_len-1)")
print(f"Batch Target Shape: {y.shape} (batch_size, seq_len-1)\n")

print("Sample 1:")
print(f"Input:  {decode_tokens(x[0].tolist())}")
print(f"Target: {decode_tokens(y[0].tolist())}")

## 2. Instantiating the Hybrid Model

Let's create the `HybridQuantumTransformerDecoder` using a noise-free simulator. We will print the number of parameters and check the model structure.

In [3]:
# Initialize the model
model = HybridQuantumTransformerDecoder(
    vocab_size=hp.vocab_size,
    embed_dim=hp.embed_dim,
    seq_len=hp.seq_len,
    num_heads=hp.num_heads,
    num_quantum_heads=hp.num_quantum_heads,
    num_qubits=hp.num_qubits,
    q_depth=hp.q_depth,
    ffn_dim=hp.ffn_dim,
    num_layers=hp.num_layers,
    use_noisy_simulator=False,  # Noise-free simulation for the notebook
    dropout=hp.dropout
)

print(f"Trainable Parameters: {count_parameters(model)}")
print(model)

## 3. Forward Pass & Attention Check

Let's feed a batch of tokens into the model to verify shape consistency and inspect the returned self-attention weight matrices.

In [4]:
model.eval()
with torch.no_grad():
    logits, attn_weights_list = model(x)

print(f"Output Logits Shape: {logits.shape} (batch_size, seq_len-1, vocab_size)")
# attn_weights_list contains the attention weights for each layer
weights = attn_weights_list[0] # Layer 0 attention weights
print(f"Attention Weights Shape: {weights.shape} (batch_size, num_heads, seq_len-1, seq_len-1)")

## 4. Training the Hybrid Model

Let's train the model for multiple epochs, tracking both training and validation loss!

In [5]:
# Training setup
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)
num_epochs = 10

# Lists to track metrics
train_losses = []
val_losses = []

print(f"Training the Hybrid model for {num_epochs} epochs...")
model.train()
for epoch in range(num_epochs):
    # Training phase
    model.train()
    total_train_loss = 0.0
    for bx, by in train_loader:
        optimizer.zero_grad()
        logits, _ = model(bx)
        loss = criterion(logits.view(-1, logits.size(-1)), by.view(-1))
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item() * bx.size(0)
    avg_train_loss = total_train_loss / len(train_loader.dataset)
    train_losses.append(avg_train_loss)
    
    # Validation phase
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for bx, by in val_loader:
            logits, _ = model(bx)
            loss = criterion(logits.view(-1, logits.size(-1)), by.view(-1))
            total_val_loss += loss.item() * bx.size(0)
    avg_val_loss = total_val_loss / len(val_loader.dataset)
    val_losses.append(avg_val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

### Training/Validation Loss Plot

In [6]:
# Plot training and validation losses
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label="Training Loss", color="royalblue", linewidth=2)
plt.plot(val_losses, label="Validation Loss", color="darkorange", linewidth=2, linestyle="--")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("Training and Validation Losses for Hybrid Quantum Transformer")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Attention Visualization

We visualize the self-attention weights. Since our hybrid model has:
- **Head 0**: Classical Head (processed via standard linear self-attention)
- **Heads 1-3**: Quantum Heads (processed via Variational Quantum Circuit feature extractor)

We can plot all 4 attention maps for a single sequence. We expect to see strictly lower-triangular attention maps due to causal masking.

In [7]:
model.eval()
with torch.no_grad():
    # Use the first batch from training
    _, attn_weights = model(x)

# Extract attention weights for the first sample in the batch
# Shape: [num_heads, seq_len-1, seq_len-1]
sample_attn = attn_weights[0][0].cpu().numpy()

fig, axes = plt.subplots(1, 4, figsize=(24, 5))

# Head 0: Classical
im0 = axes[0].imshow(sample_attn[0], cmap="viridis", vmin=0, vmax=1)
axes[0].set_title("Head 0: Classical Self-Attention", fontsize=12)
axes[0].set_xlabel("Key Position", fontsize=10)
axes[0].set_ylabel("Query Position", fontsize=10)
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

# Heads 1-3: Quantum
cmaps = ["plasma", "inferno", "magma"]
for i in range(3):
    ax = axes[i+1]
    im = ax.imshow(sample_attn[i+1], cmap=cmaps[i], vmin=0, vmax=1)
    ax.set_title(f"Head {i+1}: Quantum Self-Attention", fontsize=12)
    ax.set_xlabel("Key Position", fontsize=10)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 6. Discussion of Observations

- **Causal Masking**: The upper triangular section of all 4 attention maps is strictly 0. This confirms tokens do not attend to future tokens, validating our masking logic.
- **Loss Convergence**: Both training and validation losses decrease, confirming the model is learning!
- **Attention Patterns**: The 3 quantum heads develop features that reflect non-linear transformations from the Variational Quantum Circuits. They act as non-linear kernels, extracting distinct similarity weights compared to the linear projection in the classical head, providing the model with a richer set of representation features.